<a href="https://colab.research.google.com/github/shivamkachhadiya/Python_For_Finance_Trading/blob/main/Basic_TradingBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [219]:
from datetime import datetime
import yfinance as yf
import time

In [220]:
class MySmaTradingStrategy:
  def __init__(self,name):
    self.__name=name

  def generate_signal(self,price_data):
    print("this method is intended to be over ridded")
    return "Hold"

  @property
  def name(self):
    return self.__name

In [221]:
class MySmaTradingStrategy(MySmaTradingStrategy):
  def __init__(self,swindow,lwindow):
    self.__swindow=swindow
    self.__lwindow=lwindow
    super().__init__("My SMA Trading Strategy")


  def generate_signal(self,price_data):
    if len(price_data[-self.__lwindow:])<self.__lwindow:
      return "Hold "

    short_avg=sum(price_data[-self.__swindow:])/self.__swindow
    long_avg=sum(price_data[-self.__lwindow:])/self.__lwindow

    if short_avg>long_avg:
      return "Buy"
    elif short_avg<long_avg:
      return "Sell"
    else:
      return "Hold"

  @property
  def swindow(self):
    return self.__swindow

  @property
  def lwindow(self):
    return self.__lwindow


In [222]:
class MyTrade:
  def __init__(self,strategy_name,signal,amount):
    self.__strategy_name=strategy_name
    self.__signal=signal
    self.__amount=amount
    self.__timestamp=datetime.now()
  def execute(self):
    print(f"Executed {self.__signal} trade with strategy {self.__strategy_name} at the amount {self.__amount} at the time {self.__timestamp}")

  @property
  def strategy_name(self):
    return self.__strategy_name

  @property
  def signal(self):
    return self.__signal

  @property
  def amount(self):
    return self.__amount

  @property
  def timestamp(self):
    return self.__timestamp

In [223]:
class MockTradingAPI:
  def __init__(self,balance):
    self.__balance=balance

  def place_order(self,trade,price):
    if trade.signal=='Buy' and self.__balance>=trade.amount*price:
      self.__balance-=trade.amount*price
      print(f"placed a  buy Trade {price} , Remaingin Balance : {self.__balance} ")
    elif trade.signal=='Sell':
      self.__balance+=trade.amount*price
      print(f"placed a  sell Trade {price} , Remaingin Balance : {self.__balance} ")
    else:
      print("Insufficient Balance or Invalid Signal")

  def get_balance(self):
    return self.__balance

In [224]:

class MyTradingSystem:
    def __init__(self, api, strategy, symbol):
        self.__api = api
        self.__strategy = strategy
        self.__symbol = symbol
        self.__price_data = []

    def fetch_price_data(self):
        data = yf.download(
            tickers=self.__symbol,
            period="1d",
            interval="1m",
            auto_adjust=True,
            progress=False
        )

        if not data.empty:
            price = data["Close"].iloc[-1]

            # Handle yfinance returning a Series
            if hasattr(price, "item"):
                price = price.item()

            self.__price_data.append(float(price))

            if len(self.__price_data) > self.__strategy.lwindow:
                self.__price_data.pop(0)

            print(f"Fetched new price data: {price}")
            return float(price)

        else:
            print("No data fetched")
            return None

    def run(self):
        price = self.fetch_price_data()

        if price is None:
            return

        signal = self.__strategy.generate_signal(self.__price_data)
        print(f"Generated signal: {signal}")

        if signal in ["Buy", "Sell"]:
            trade = MyTrade(self.__strategy.name, signal, 1)
            trade.execute()
            self.__api.place_order(trade, price)

    @property
    def api(self):
        return self.__api

    @property
    def strategy(self):
        return self.__strategy

    @property
    def symbol(self):
        return self.__symbol


In [225]:
symbol="AAPL"
api=MockTradingAPI(balance=1000)
strategy=MySmaTradingStrategy(3,5)
trading_system=MyTradingSystem(api,strategy,symbol)

for _ in range(2):
  trading_system.run()
  print(f"Remaing Balance :{api.get_balance()}")
  time.sleep(2)

Fetched new price data: 342.41290283203125
Generated signal: Hold 
Remaing Balance :1000
Fetched new price data: 342.3450012207031
Generated signal: Hold 
Remaing Balance :1000


In [226]:
# Execute the strategy
strategy = MySmaTradingStrategy(3, 5)

price_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
signal = strategy.generate_signal(price_data)

print("Signal:", signal)

trade = MyTrade(strategy_name=strategy.name,signal=signal,amount=100)

api = MockTradingAPI(100000)
api.place_order(trade, 100)

print("Final balance:", api.get_balance())


Signal: Buy
placed a  buy Trade 100 , Remaingin Balance : 90000 
Final balance: 90000
